<a href="https://colab.research.google.com/github/jed-aloui/huffman-compressor/blob/main/huffman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
from collections import Counter


class Node:

    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    # Définition des boutons pour la file de priorité (min-heap)
    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_tree(text):
    """Construit l'arbre de Huffman à partir du texte."""
    # 1. Compter la fréquence de chaque caractère
    frequency = Counter(text)

    # 2. Créer une file de priorité avec des nœuds de feuilles
    heap = [Node(char, freq) for char, freq in frequency.items()]
    heapq.heapify(heap)

    # 3. Fusionner les nœuds jusqu'à ce qu'il n'en reste qu'un seul (la racine)
    while len(heap) > 1:
        node1 = heapq.heappop(heap)
        node2 = heapq.heappop(heap)

        merged = Node(None, node1.freq + node2.freq)
        merged.left = node1
        merged.right = node2

        heapq.heappush(heap, merged)

    return heap[0] if heap else None


def generate_codes(node, current_code="", codes=None):
    """Génère les codes binaires pour chaque caractère en parcourant l'arbre."""
    if codes is None:
        codes = {}

    if node is None:
        return codes

    # Si c'est une feuille, on enregistre son code
    if node.char is not None:
        codes[node.char] = current_code

    generate_codes(node.left, current_code + "0", codes)
    generate_codes(node.right, current_code + "1", codes)

    return codes


def encode(text, codes):
    """Compresse le texte en remplaçant les caractères par leurs nouveaux codes."""
    return "".join(codes[char] for char in text)


def decode(encoded_text, root):
    """Décompresse le texte binaire en parcourant l'arbre de Huffman."""
    decoded_text = []
    current_node = root

    for bit in encoded_text:
        if bit == "0":
            current_node = current_node.left
        else:
            current_node = current_node.right

        # Si on atteint une feuille, on récupère le caractère
        if current_node.char is not None:
            decoded_text.append(current_node.char)
            current_node = root

    return "".join(decoded_text)


# --- Exemple d'utilisation ---
if __name__ == "__main__":
    texte_original = "les algorithmes de huffman sont fascinants !"
    print(f"Texte Original : {texte_original}")

    # Étape 1 : Construction de l'arbre
    arbre = build_huffman_tree(texte_original)

    # Étape 2 : Génération des codes uniques
    dictionnaire_codes = generate_codes(arbre)
    print("\n--- Codes de Huffman générés ---")
    for car, code in sorted(dictionnaire_codes.items(), key=lambda x: len(x[1])):
        print(f"'{car}': {code}")

    # Étape 3 : Encodage
    texte_compresse = encode(texte_original, dictionnaire_codes)
    print(f"\nTexte Compressé (Binaire) : {texte_compresse}")

    # Étape 4 : Décodage (Vérification)
    texte_decompile = decode(texte_compresse, arbre)
    print(f"Texte Décompressé : {texte_decompile}")


Texte Original : les algorithmes de huffman sont fascinants !

--- Codes de Huffman générés ---
's': 011
' ': 100
'i': 0000
'h': 0010
'm': 0101
'f': 1010
'e': 1011
't': 1100
'a': 1110
'n': 1111
'u': 00010
'd': 00011
'g': 00110
'!': 00111
'c': 01000
'r': 01001
'o': 11010
'l': 11011

Texte Compressé (Binaire) : 1101110110111001110110110011011010010010000110000100101101101110000011101110000100001010101010010111101111100011110101111110010010101110011010000000111111101111110001110000111
Texte Décompressé : les algorithmes de huffman sont fascinants !
